# Two-speaker split for the ADReSSo dataset

Notebook version of `split_speakers_pipeline.py`, run over every `.wav` in the dataset.

```
Two-speaker split with overlap removal for ADReSSo recordings (e.g. adrso025.wav).

Pipeline
  0. Stereo check      -> are the channels spatially different? (Pearson r, RMS ratio)
  1. Preprocess        -> mono downmix (mean of channels), resample 44.1 kHz -> 16 kHz
  2. Diarization       -> pyannote segmentation-3.0 (ONNX) + WeSpeaker ResNet34-LM embeddings
                          + agglomerative clustering (cosine threshold 0.7), via sherpa-onnx
  3. Role verification -> Whisper base.en (ONNX, int8) transcribes every segment
  4. Overlap detection -> intersection of segments that belong to different speakers
  5. Output            -> one time-aligned file per speaker; the other speaker and all
                          overlap regions are muted (150 ms padding, 10 ms fades)

Setup
  pip install sherpa-onnx soundfile librosa numpy
  (models are downloaded automatically from the sherpa-onnx GitHub releases)

Usage
  python split_speakers_pipeline.py adrso025.wav --out outputs/
```

## Setup

In [ ]:
%pip install -q sherpa-onnx soundfile librosa numpy

In [ ]:
import argparse, csv, os, tarfile, urllib.request
import numpy as np, soundfile as sf, librosa, sherpa_onnx

REL = "https://github.com/k2-fsa/sherpa-onnx/releases/download"
MODELS = {
    "seg":  (f"{REL}/speaker-segmentation-models/sherpa-onnx-pyannote-segmentation-3-0.tar.bz2",
             "sherpa-onnx-pyannote-segmentation-3-0/model.onnx"),
    "emb":  (f"{REL}/speaker-recongition-models/wespeaker_en_voxceleb_resnet34_LM.onnx",  # "recongition" is the real tag name
             "wespeaker_en_voxceleb_resnet34_LM.onnx"),
    "asr":  (f"{REL}/asr-models/sherpa-onnx-whisper-base.en.tar.bz2",
             "sherpa-onnx-whisper-base.en"),
}
SR16 = 16000

## Pipeline functions (unchanged from the script)

In [ ]:
def fetch(key, mdir):
    url, rel = MODELS[key]
    path = os.path.join(mdir, rel)
    if not os.path.exists(path):
        os.makedirs(mdir, exist_ok=True)
        dst = os.path.join(mdir, os.path.basename(url))
        urllib.request.urlretrieve(url, dst)
        if dst.endswith(".tar.bz2"):
            with tarfile.open(dst) as t:
                t.extractall(mdir)
    return path


def stereo_check(x):
    if x.ndim == 1:
        return "mono file"
    L, R = x[:, 0], x[:, 1]
    r = np.corrcoef(L, R)[0, 1]
    lr_db = 20 * np.log10(np.sqrt((L**2).mean()) / np.sqrt((R**2).mean()))
    return f"channel correlation r={r:.3f}, L/R level={lr_db:+.1f} dB"


def diarize(a16, mdir, threshold=0.7):
    cfg = sherpa_onnx.OfflineSpeakerDiarizationConfig(
        segmentation=sherpa_onnx.OfflineSpeakerSegmentationModelConfig(
            pyannote=sherpa_onnx.OfflineSpeakerSegmentationPyannoteModelConfig(model=fetch("seg", mdir))),
        embedding=sherpa_onnx.SpeakerEmbeddingExtractorConfig(model=fetch("emb", mdir)),
        clustering=sherpa_onnx.FastClusteringConfig(threshold=threshold),  # or num_clusters=2
        min_duration_on=0.2,   # drop speech islands shorter than 0.2 s
        min_duration_off=0.3,  # fill pauses shorter than 0.3 s
    )
    sd = sherpa_onnx.OfflineSpeakerDiarization(cfg)
    return [(s.start, s.end, s.speaker) for s in sd.process(a16).sort_by_start_time()]


def transcribe(a16, segs, mdir):
    d = fetch("asr", mdir)
    rec = sherpa_onnx.OfflineRecognizer.from_whisper(
        encoder=f"{d}/base.en-encoder.int8.onnx", decoder=f"{d}/base.en-decoder.int8.onnx",
        tokens=f"{d}/base.en-tokens.txt", language="en")
    out = []
    for p, q, _ in segs:
        s = rec.create_stream()
        s.accept_waveform(SR16, a16[int(p * SR16):int(q * SR16)])
        rec.decode_stream(s)
        out.append(s.result.text.strip())
    return out


def overlaps(segs):
    ov = []
    for i, (p, q, c) in enumerate(segs):
        for p2, q2, c2 in segs[i + 1:]:
            if c != c2 and p2 < q and q2 > p:
                ov.append((max(p, p2), min(q, q2)))
    return ov


def speaker_mask(segs, spk, n, sr, pad=0.15):
    m = np.zeros(n, bool)
    for p, q, c in segs:                      # own speech + padding
        if c == spk:
            m[int(max(0, p - pad) * sr):int(min(n / sr, q + pad) * sr)] = True
    for p, q, c in segs:                      # remove other speaker -> also removes overlaps
        if c != spk:
            m[int(p * sr):int(q * sr)] = False
    return m


def fade(m, sr, ms=10):
    k = int(sr * ms / 1000)
    return np.convolve(m.astype(np.float32), np.ones(k) / k, "same")

## Dataset configuration

Set `DATA_ROOT` to the ADReSSo folder on the server. All `.wav` files under it are processed recursively, and outputs mirror the input folder structure under `OUT_ROOT` (e.g. `diagnosis/train/audio/ad/adrso025.wav` -> `OUT_ROOT/diagnosis/train/audio/ad/adrso025_participant.wav`).

In [ ]:
DATA_ROOT     = "/path/to/ADReSSo21"      # <- change to the dataset location on the server
OUT_ROOT      = "outputs"
MODELS_DIR    = "models"
THRESHOLD     = 0.7
NO_ASR        = False
SKIP_EXISTING = True                     # resume: skip files whose outputs already exist

## Per-file processing

The body of `main()` from the script, with the command-line arguments replaced by function parameters.

In [ ]:
from types import SimpleNamespace


def process_file(wav, out, models=MODELS_DIR, threshold=THRESHOLD, no_asr=NO_ASR):
    args = SimpleNamespace(wav=wav, out=out, models=models, threshold=threshold, no_asr=no_asr)
    os.makedirs(args.out, exist_ok=True)
    base = os.path.splitext(os.path.basename(args.wav))[0]

    x, sr = sf.read(args.wav)
    print("[0]", stereo_check(x))

    mono = x.mean(1) if x.ndim == 2 else x
    a16 = librosa.resample(mono.astype(np.float32), orig_sr=sr, target_sr=SR16)

    segs = diarize(a16, args.models, args.threshold)
    spk_time = {}
    for p, q, c in segs:
        spk_time[c] = spk_time.get(c, 0) + (q - p)
    print("[2] speakers found:", {f"spk{k}": round(v, 1) for k, v in spk_time.items()})
    if len(spk_time) != 2:
        print("    WARNING: not 2 clusters - tune --threshold or use num_clusters=2")

    # Role heuristic for ADReSSo: participant talks most. VERIFY with the transcripts.
    ranked = sorted(spk_time, key=spk_time.get, reverse=True)
    roles = {ranked[0]: "participant", ranked[1]: "interviewer"}

    texts = [""] * len(segs) if args.no_asr else transcribe(a16, segs, args.models)
    ov = overlaps(segs)
    print(f"[4] {len(ov)} overlap regions, {sum(q - p for p, q in ov):.2f} s total")

    with open(os.path.join(args.out, f"{base}_segments.csv"), "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["start", "end", "cluster", "role", "overlap", "transcript"])
        for (p, q, c), t in zip(segs, texts):
            is_ov = any(a < q and b > p for a, b in ov)
            w.writerow([f"{p:.2f}", f"{q:.2f}", f"spk{c}", roles.get(c, "?"), is_ov, t])
            print(f"    {p:7.2f}-{q:7.2f} {roles.get(c, '?'):11s} {'OV' if is_ov else '  '} {t}")

    for spk, role in roles.items():
        g = fade(speaker_mask(segs, spk, len(x), sr), sr)
        y = x * (g[:, None] if x.ndim == 2 else g)
        sf.write(os.path.join(args.out, f"{base}_{role}.wav"), y, sr, subtype="PCM_16")
    print("[5] written to", args.out)

## Run over the dataset

In [ ]:
import glob, traceback

# download all models once up front
for k in MODELS:
    fetch(k, MODELS_DIR)

wavs = sorted(glob.glob(os.path.join(DATA_ROOT, "**", "*.wav"), recursive=True))
print(f"{len(wavs)} wav files found under {DATA_ROOT}")

failed = []
for i, wav in enumerate(wavs, 1):
    rel_dir = os.path.relpath(os.path.dirname(wav), DATA_ROOT)
    out = os.path.join(OUT_ROOT, rel_dir)
    base = os.path.splitext(os.path.basename(wav))[0]
    if SKIP_EXISTING and os.path.exists(os.path.join(out, f"{base}_segments.csv")):
        print(f"[{i}/{len(wavs)}] skip {wav}")
        continue
    print(f"\n[{i}/{len(wavs)}] {wav}")
    try:
        process_file(wav, out)
    except Exception as e:
        traceback.print_exc()
        failed.append((wav, repr(e)))

print(f"\nDone. {len(wavs) - len(failed)} ok, {len(failed)} failed")
for w, e in failed:
    print("  FAILED:", w, e)